# UU (Uppsala) datasets — data origin and how to read them

The datasets uploaded to NOMAD from this project come from
[github.com/MaMMoS-project/uppsala-data](https://github.com/MaMMoS-project/uppsala-data)
(layout `RSPt/` + `UppASD/`, one folder per composition, e.g. `Al2Fe6C`).
The derived files in each dataset — `intrinsic_properties.yaml` (mammos yaml v2:
`Js`, `Ms`, `Ku`, `Tc` with EMMO ontology labels/IRIs) and `UppASD/MC_*/thermal.csv`
(mammos csv v3: temperature-dependent `T, Ms, Js, E, Cv, chi, U_L`) — are **generated
and validated with [mammos-parser](https://github.com/MaMMoS-project/mammos-parser)**
(`mammos-parser uppsala-data generate-derived-files <dir>` / `validate-dataset <dir>`),
not by hand.

This upload contains:
- `<dataset>.h5` — a **mammos-entity `EntityCollection`** (sub-collections
  `intrinsic_properties`, `thermal_MC_<n>` per MC run, `raw_data` with the full folder tree),
- `<dataset>.archive.yaml` — the NOMAD entry (quantities linked to the h5 via
  `m_annotations.hdf5.path`, plus a repeating `thermal_properties` subsection with the
  temperature-dependent curves),
- `cif_structure_<dataset>.cif` + `datasets/<dataset>/...` — raw data / provenance,
- `packages/` — the conversion/validation code used to build the upload,
- `<dataset>_README.md` — a short README of this upload.

The cells below use **local paths only** (the files next to this notebook in the
unpacked upload), so the same notebook works unchanged for every dataset upload.


## Reading the data with `mammos_entity` (`me`)

Simplest: read the uploaded `.h5` directly (it is a mammos-entity collection file).


In [ ]:
import glob
import mammos_entity as me

# Exactly one .h5 sits at the root of this upload, next to this notebook.
# Derive the dataset name from it instead of hardcoding it.
h5_files = glob.glob('*.h5')
assert len(h5_files) == 1, f'expected one .h5 next to the notebook, found {h5_files}'
dataset = h5_files[0][:-3]            # e.g. 'Al2Fe6C'
print('dataset:', dataset)
col = me.from_hdf5(h5_files[0])       # mammos-entity EntityCollection
col


In [ ]:
# Intrinsic scalar properties with ontology metadata (label, IRI, unit, description)
ip = col.intrinsic_properties
ip.Js, ip.Ms, ip.Ku, ip.Tc            # e.g. Js = 1.145 T, label SpontaneousMagneticPolarization


In [ ]:
# Temperature-dependent curves: one sub-collection per MC run (thermal_MC_1, ...)
mc_names = sorted(n for n in dir(col) if n.startswith('thermal_MC_'))
th = getattr(col, mc_names[0])
print('MC runs in this dataset:', mc_names)
th.T.value[:5], th.Js.value[:5]       # arrays of T [K] and Js [T]


In [ ]:
# Alternatively read the raw mammos files under datasets/<dataset>/ directly:
props = me.from_yaml(f'datasets/{dataset}/intrinsic_properties.yaml')
thermal = me.from_csv(f'datasets/{dataset}/UppASD/MC_1/thermal.csv')
props.Js, thermal.T.value[:5], thermal.Js.value[:5]


---

This notebook ships in every UU upload zip (`Evaluation_dataset_UU.ipynb`) and works
with the local files of the unpacked upload it sits in. The upload pipeline itself is
`upload_UU_DFT+MC` (see `packages/`); datasets are validated/generated with mammos-parser.
